# Sparsity + Quantization 조합 (로컬 버전)

## 개요
Pruning(가지치기)과 Quantization(양자화)을 조합하여 더 공격적인 압축을 시도합니다.

### 실행 전 필수 사항
```bash
cd lg-aimers8-llm-compression
source venv/bin/activate
jupyter notebook
```

---

# 1. Import 및 환경 확인

In [1]:
import os
import sys
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor.modifiers.pruning import SparseGPTModifier

print("=" * 60)
print("환경 정보")
print("=" * 60)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU 없음 - CPU로 실행됩니다")
print("=" * 60)

환경 정보
Python: 3.10.13
PyTorch: 2.9.1
CUDA 사용 가능: False
⚠️  GPU 없음 - CPU로 실행됩니다


# 2. 하이퍼파라미터 설정

In [2]:
# ============================================================================
# 모델 설정
# ============================================================================
# 로컬 모델 경로 (다운로드 불필요!)
MODEL_ID = "./open/base_model"
OUT_DIR = "./model_sparse_quant"

# 데이터셋 설정
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# 캘리브레이션 설정 (환경에 따라 자동 조절)
if torch.cuda.is_available():
    NUM_CALIBRATION_SAMPLES = 512
    MAX_SEQUENCE_LENGTH = 1024
else:
    NUM_CALIBRATION_SAMPLES = 128  # CPU용 더 축소
    MAX_SEQUENCE_LENGTH = 512

# ============================================================================
# Sparsity 설정
# ============================================================================
SPARSITY_RATIO = 0.5  # 50% 가중치 제거

# 양자화 설정
QUANT_SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]
GROUP_SIZE = 128

# 원본 모델 크기
ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("Sparsity + Quantization 설정")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"SPARSITY_RATIO: {SPARSITY_RATIO} ({SPARSITY_RATIO*100}% 희소화)")
print(f"QUANT_SCHEME: {QUANT_SCHEME}")
print(f"NUM_CALIBRATION_SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print("=" * 60)

Sparsity + Quantization 설정
MODEL_ID: ./open/base_model
SPARSITY_RATIO: 0.5 (50.0% 희소화)
QUANT_SCHEME: W4A16
NUM_CALIBRATION_SAMPLES: 128


# 3. 모델 로드

In [3]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

device_map = "auto" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32 if not torch.cuda.is_available() else torch.bfloat16,
    trust_remote_code=True,
    device_map=device_map,
)

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print(f"[INFO] 디바이스: {device_map}")
print("[INFO] 모델 로드 완료")

`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델 로드 중...
[INFO] 모델 파라미터: 1,279,391,488
[INFO] 디바이스: cpu
[INFO] 모델 로드 완료


# 4. 데이터셋 로드

In [4]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")

[INFO] 캘리브레이션 데이터 로드 중...


Map:   0%|          | 0/128 [00:00<?, ? examples/s]

[INFO] 데이터셋 크기: 128


# 5. SparseGPT + GPTQ 압축

⚠️ **주의**: 이 과정은 CPU에서 매우 오래 걸립니다 (4-8시간+)

In [5]:
print("[INFO] SparseGPT + GPTQ 압축 시작")
print(f"  - Sparsity: {SPARSITY_RATIO*100}%")
print(f"  - Quantization: {QUANT_SCHEME}")

if torch.cuda.is_available():
    print("\n🚀 GPU 모드: 30-60분 예상\n")
else:
    print("\n⏳ CPU 모드: 4-8시간+ 예상 (매우 느림)\n")

recipe = [
    # Step 1: SparseGPT 희소화
    SparseGPTModifier(
        sparsity=SPARSITY_RATIO,
        targets=TARGETS,
        sequential_update=True,
    ),
    
    # Step 2: GPTQ 양자화
    GPTQModifier(
        scheme=QUANT_SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=GROUP_SIZE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] 압축 완료!")

[INFO] SparseGPT + GPTQ 압축 시작
  - Sparsity: 50.0%
  - Quantization: W4A16

⏳ CPU 모드: 4-8시간+ 예상 (매우 느림)



Tokenizing:   0%|          | 0/128 [00:00<?, ? examples/s]

2026-02-09T21:55:10.688955+0900 | reset | INFO - Compression lifecycle reset
2026-02-09T21:55:10.690512+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-09T21:55:10.724635+0900 | initialize | INFO - Compression lifecycle initialized for 2 modifiers
2026-02-09T21:55:10.725155+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `SparseGPTModifier`
2026-02-09T21:55:10.731553+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead
2026-02-09T21:55:10.731918+0900 | get_sequential_targets | WARNING - Passing sequential targets through modifiers is deprecated, please use `oneshot(sequential_targets=...)`


W0209 21:55:10.765000 69210 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:28<00:00,  4.50it/s]

2026-02-09T21:55:39.326778+0900 | compress_modules | INFO - Sparsifying model.layers.0.self_attn.q_proj. using 128 samples


...

(30/31): Calibrating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:18<00:00,  6.82it/s]

2026-02-09T22:23:29.738035+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 128 samples


2026-02-09T22:23:30.118447+0900 | compress | METRIC - time 0.38s
2026-02-09T22:23:30.118872+0900 | compress | METRIC - error 1622.86
2026-02-09T22:23:30.119758+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-09T22:23:30.119994+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-09T22:23:30.121504+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 128 samples
2026-02-09T22:23:30.392897+0900 | compress | METRIC - time 0.27s
2026-02-09T22:23:30.393307+0900 | compress | METRIC - error 450.82
2026-02-09T22:23:30.394167+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-09T22:23:30.394377+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-09T22:23:30.395126+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 128 samples
2026-02-09T22:23:30.614719+0900 | compress | METRIC - time 0.22s
2026-02-09T22:2

(31/31): Propagating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:00<00:00, 4120.77it/s]

2026-02-09T22:23:37.372220+0900 | finalize | INFO - Compression lifecycle finalized for 2 modifiers
2026-02-09T22:23:37.377064+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`

[INFO] 압축 완료!


# 6. 모델 저장 및 크기 비교

In [8]:
print("[INFO] 모델 저장 중...")

os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 파일 확인
print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("모델 크기 비교")
print("=" * 60)
print(f"  원본 모델:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  압축 모델:     {quantized_size_gb:.2f} GB")
print(f"  ----------------------------------------")
print(f"  크기 감소:     {ORIGINAL_MODEL_SIZE_GB - quantized_size_gb:.2f} GB")
print(f"  압축률:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print(f"  압축 배수:     {ORIGINAL_MODEL_SIZE_GB / quantized_size_gb:.2f}x")
print("=" * 60)

[INFO] 모델 저장 중...


Calculating model sparsity: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 753/753 [00:00<00:00, 1622.93it/s]
Compressing model: 212it [00:00, 2084.75it/s]



[INFO] 저장된 파일:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  merges.txt: 1.2 MB
  model.safetensors: 1407.7 MB
  recipe.yaml: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 7.9 MB
  tokenizer_config.json: 0.1 MB
  vocab.json: 1.9 MB

모델 크기 비교
  원본 모델:     2.56 GB
  압축 모델:     1.42 GB
  ----------------------------------------
  크기 감소:     1.14 GB
  압축률:        55.4%
  압축 배수:     1.80x


# 7. 제출 파일 생성

In [9]:
zip_name = "submit_sparse_quant"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("✅ 용량 제한 충족 (≤ 10GB)")
else:
    print("❌ 용량 초과!")

print(f"\n📁 파일 위치: {os.path.abspath(f'{zip_name}.zip')}")

[INFO] submit_sparse_quant.zip 생성 중...
[INFO] 생성 완료: submit_sparse_quant.zip (0.88 GB)
✅ 용량 제한 충족 (≤ 10GB)

📁 파일 위치: /Users/imdonghyeon/Desktop/lg-aimers8-llm-compression/submit_sparse_quant.zip


# 8. 모델 테스트

In [ ]:
print("[INFO] 압축된 모델 테스트...")

message = [{"role": "user", "content": "1 + 1 = ?"}]

input_ids = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

output = model.generate(
    input_ids,
    max_new_tokens=50,
    do_sample=False,
)

response = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"\n응답:\n{response}")